# 주식관련사채(CB/BW/EB) 종합 조회

종목코드 하나를 입력하면 **3가지 소스**에서 사채 정보를 수집하여 비교합니다.

| 섹션 | 데이터 소스 | 설명 |
|------|:----------:|------|
| **[1] 예탁원 현황** | Seibro API | 실시간 등록 현황 (전체 칼럼) |
| **[2] 주요사항보고서** | DART 공시 API | CB/BW/EB 발행결정 공시 이력 (최근 5년) |
| **[3] 정기보고서** | DART 사업/반기보고서 XML | 보고서에 기재된 CB/BW 미상환 현황 |

## 0. 종목코드 입력

In [1]:
!pip install git+https://github.com/beaten-by-the-market/seibro-api.git

  Cloning https://github.com/beaten-by-the-market/seibro-api.git to c:\users\peter\appdata\local\temp\pip-req-build-epclf2ke
  Resolved https://github.com/beaten-by-the-market/seibro-api.git to commit e51dc83560c8c59858f27a4830bcf9468a430abd
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seibro-api: filename=seibro_api-0.1.0-py3-none-any.whl size=17956 sha256=e9a19ebd94eec9a87b943b47607907060a07ca5efa07359e35f37e9473ca2930
  Stored in directory: C:\Users\Peter\AppData\Local\Temp\pip-ephem-wheel-cache-q5kez9di\wheels\9d\59\a6\afb6d23ba585a05a812b450206aabae0a95bcb9ce7af77fb72
Successfully built seibro-api


  Running command git clone --filter=blob:none --quiet https://github.com/beaten-by-the-market/seibro-api.git 'C:\Users\Peter\AppData\Local\Temp\pip-req-build-epclf2ke'

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## API 키 설정

아래 셀에 본인의 API 키를 입력하세요.

- **SEIBRO_API_KEY**: [한국예탁결제원 Open API](http://seibro.or.kr) 에서 발급
- **DART_API_KEY**: [DART 오픈API](https://opendart.fss.or.kr) 에서 발급

In [2]:
# 조회할 종목코드를 입력하세요
STOCK_CODE = "307750"  # 국전약품 (CB + BW 보유)

In [3]:
import pandas as pd
from IPython.display import display, HTML
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', None)

---
## [1] 예탁원(Seibro) 주식관련사채 현황

한국예탁결제원 Open API에서 해당 종목의 **CB/BW/EB 등록 현황**을 실시간으로 조회합니다.

- 데이터 소스: `getXrcStkStatInfo` + `getBondStatInfo`
- 채권 상세(발행금액, 미상환잔액, 전환/행사가, 신용등급 등) + 전환가능 주식수 자동 계산
- **이 데이터가 메인 정보**입니다.

In [4]:
from seibro_api import get_stock_bonds

df_seibro = get_stock_bonds(STOCK_CODE, save_csv=False)

if not df_seibro.empty:
    print(f"총 {len(df_seibro)}건")
    display(df_seibro)
else:
    print("해당 종목에 주식관련사채가 없습니다.")


[1/4] 종목 307750의 표준코드 조회 중...


  -> 국전약품 (고객번호: 33915, ISIN: KR7307750000)

[2/4] 주식관련사채(CB/BW/EB) 목록 조회 중...


  -> 1건의 사채 발견
     국전약품3BW(사모/신/풋) (비분리형BW)

[3/4] 사채별 상세정보 조회 중...


  OK KR6307751E82

[4/4] 데이터 통합 및 정리 중...

  국전약품(307750) 주식관련사채 조회 결과
  총 1건

  [1] 국전약품3BW(사모/신/풋)
      유형: 비분리형BW | 신주인수권(BW) | 매출
      기간: 20240827 ~ 20290827
      미상환잔액: 28420450000 | 전환/행사가: 3630 | 전환가능주식수: 7829325
총 1건


,채권코드,채권명,CB/BW/EB,주권코드,주식종목명,채권종류,모집방법,발행방법,발행일,만기일,발행금액,미상환잔액,표면이자율,전환/행사가,행사비율,전환가능_주식수,옵션,강제조기상환,금리변동,보증,순위,이자지급방법,원금상환방법,만기보장수익율,만기상환율,KIS_등급,NICE_등급,KR_등급,상장일,상장폐지일,전자증권여부
0,KR6307751E82,국전약품3BW(사모/신/풋),비분리형BW,KR7307750000,국전약품,신주인수권(BW),매출,전액등록,20240827,20290827,43000000000,28420450000,0,3630,100,7829325,PUT,NO,고정,무보증,선순위,이표,만기상환,0,100,,,,,,전자등록


---
## [2] DART 주요사항보고서 이력 (보조)

DART 전자공시에서 최근 5년간 **CB/BW/EB 발행결정 공시**를 조회합니다.

- 데이터 소스: `dart.event()` (전환사채발행 / 신주인수권부사채발행 / 교환사채발행)
- 이사회 결의일, 납입일, 사채만기일, 발행방법 등 공시 시점의 정보
- 공시URL 클릭 시 DART 공시 원문으로 이동

In [5]:
from seibro_api import get_dart_cb_events

df_dart = get_dart_cb_events(STOCK_CODE, save_csv=False)

if not df_dart.empty:
    # 공시URL 생성
    if "접수번호" in df_dart.columns:
        df_dart["공시URL"] = df_dart["접수번호"].apply(
            lambda x: f'<a href="https://dart.fss.or.kr/dsaf001/main.do?rcpNo={x}" target="_blank">공시보기</a>'
        )

    # 주요 칼럼만 표출
    display_cols = [
        "회사명", "사채종류_회차", "사채발행방법",
        "권면총액(원)", "이사회결의일", "납입일",
        "사채만기일", "공시유형", "공시URL",
    ]
    cols_exist = [c for c in display_cols if c in df_dart.columns]
    df_dart_display = df_dart[cols_exist]

    print(f"총 {len(df_dart_display)}건")
    display(HTML(df_dart_display.to_html(escape=False, index=False)))
else:
    print("해당 종목에 CB/BW/EB 발행 공시가 없습니다.")


[DART] 307750 주식관련사채 발행 공시 조회
  기간: 20210321 ~ 20260320 (5년)

  [CB] 전환사채발행 조회 중...


    -> 1건

  [BW] 신주인수권부사채발행 조회 중...


    -> 1건

  [EB] 교환사채발행 조회 중...


{'status': '013', 'message': '조회된 데이타가 없습니다.'}
    해당 없음

  총 2건 조회 완료 (CB/BW/EB 통합)
총 2건


회사명,사채종류_회차,사채발행방법,권면총액(원),이사회결의일,납입일,사채만기일,공시유형,공시URL
국전약품,2,사모,"65,000,000,000",2022년 09월 14일,2022년 09월 16일,2027년 09월 16일,CB,공시보기
국전약품,3,사모,"43,000,000,000",2024년 08월 23일,2024년 08월 27일,2029년 08월 27일,BW,공시보기


---
## [3] DART 정기보고서 이력 (보조)

최근 **사업보고서 + 반기보고서** XML 원문에서 CB/BW 테이블을 파싱합니다.

- 데이터 소스: `dart.document()` → XML 파싱 (aclass=CB, BW)
- 보고서 기준일 시점의 미상환잔액을 확인할 수 있어, **시점별 변동 비교**에 유용
- `[첨부정정]` 공시는 자동으로 원본 접수번호를 추적합니다

In [6]:
from seibro_api import get_bonds_from_report

report_results = get_bonds_from_report(STOCK_CODE, save_csv=False)

df_cb = report_results.get("cb", pd.DataFrame())
df_bw = report_results.get("bw", pd.DataFrame())

# 표출용 칼럼
report_display_cols = [
    "보고서유형", "보고서기간", "사채종류", "회차",
    "발행일", "만기일", "발행총액", "미상환잔액",
]


[1/3] 307750 최근 사업/반기보고서 조회 중...
  기간: 20240320 ~ 20260320


  -> 사업보고서: [기재정정]사업보고서 (2024.12) (20250321)
  -> 반기보고서: 반기보고서 (2025.06) (20250814)

[2/3] rcept_no_new 확보 중...
  반기보고서: 20250814002654


  사업보고서: 20250321001847



[3/3] XML 원문 다운로드 중...


  OK 반기보고서 (20250814002654) - 1,222,838자


...

  반기보고서(2025.06): 2건 (메타: {'BASE_DT': '2025년 06월 30일', 'WONSTOCK': '(단위 : 백만원, 주)'})


  사업보고서(2024.12): 2건 (메타: {'BASE_DT': '2024년 12월 31일', 'WONSTOCK': '(단위 : 백만원, 주)'})

  총 4건

[파싱] bw (aclass=BW) ...
  반기보고서(2025.06): 2건 (메타: {'BASE_DT': '2025년 06월 30일', 'WONSTOCK': '(단위 : 백만원, 주)'})


  사업보고서(2024.12): 2건 (메타: {'BASE_DT': '2024년 12월 31일', 'WONSTOCK': '(단위 : 백만원, 주)'})

  총 4건

[파싱] eb (aclass=EB) ...


  반기보고서(2025.06): 데이터 없음


  사업보고서(2024.12): 데이터 없음
  eb 데이터를 찾을 수 없습니다.


### CB (전환사채)

In [7]:
if not df_cb.empty:
    cols_exist = [c for c in report_display_cols if c in df_cb.columns]
    df_cb_display = df_cb[cols_exist].dropna(subset=["사채종류"], how="all") if "사채종류" in df_cb.columns else df_cb[cols_exist]
    print(f"CB {len(df_cb_display)}건")
    display(df_cb_display)
else:
    print("CB 해당 없음")

CB 2건


,보고서유형,보고서기간,사채종류,회차,발행일,만기일,발행총액,미상환잔액
0,반기보고서,2025.06,무기명식 무이권부무보증 사모 전환사채,제2회,2022년 09월 16일,2027년 09월 16일,"65,000",500
2,사업보고서,2024.12,무기명식 무이권부무보증 사모 전환사채,제2회,2022년 09월 16일,2027년 09월 16일,"65,000","13,400"


### BW (신주인수권부사채)

In [8]:
if not df_bw.empty:
    cols_exist = [c for c in report_display_cols if c in df_bw.columns]
    df_bw_display = df_bw[cols_exist].dropna(subset=["사채종류"], how="all") if "사채종류" in df_bw.columns else df_bw[cols_exist]
    print(f"BW {len(df_bw_display)}건")
    display(df_bw_display)
else:
    print("BW 해당 없음")

BW 2건


,보고서유형,보고서기간,사채종류,회차,발행일,만기일,발행총액,미상환잔액
0,반기보고서,2025.06,무기명식 무보증사모 신주인수권부사채,제3회,2024년 08월 27일,2029년 08월 27일,"43,000","43,000"
2,사업보고서,2024.12,무기명식 무보증사모 신주인수권부사채,제3회,2024년 08월 27일,2029년 08월 27일,"43,000","43,000"


---
## 전체 결과 요약

위에서 수집한 모든 DataFrame은 아래 변수에 저장되어 있습니다.

| 변수 | 내용 |
|------|------|
| `df_seibro` | 예탁원 실시간 현황 (전체 칼럼) |
| `df_dart` | DART CB/BW/EB 발행결정 공시 |
| `df_cb` | 정기보고서 CB 상세 (사업/반기) |
| `df_bw` | 정기보고서 BW 상세 (사업/반기) |

In [9]:
print(f"[1] 예탁원 현황: {len(df_seibro)}건")
print(f"[2] DART 주요사항보고서: {len(df_dart)}건")
print(f"[3] 정기보고서 CB: {len(df_cb)}건")
print(f"[3] 정기보고서 BW: {len(df_bw)}건")

[1] 예탁원 현황: 1건
[2] DART 주요사항보고서: 2건
[3] 정기보고서 CB: 4건
[3] 정기보고서 BW: 4건


In [ ]:
import os
os.environ["SEIBRO_API_KEY"] = "본인키를 입력하세요"
os.environ["DART_API_KEY"] = "본인키를 입력하세요"